In [ ]:
###
library(data.table)
library(dplyr)
library(survival)
library(survminer)
library(coxphw)
library(ggplot2)
library(ggpubr)
library(cowplot)
library(RNOmni)
theme_set(theme_cowplot())
library(cmprsk) # Competing Risks Regression
library(riskRegression)
library(pec)
library(prodlim)
# 
library(broom)


In [ ]:
############
### source:  https://www.biostars.org/p/80597/ and the supplement of Yang et al. Nature 2012.
INT_yang2012 <- function(x){
  y<-qnorm((rank(x,na.last='keep')-0.5)/sum(!is.na(x)))
  return(y)
}

In [ ]:
list_prs_score.afr <- system("ls /medpop/esp2/mesbah/projects/Meta_GWAS/MetaGWAS_N900k/sumHer/prs/scores/afr/*.profile | awk '{print $NF}'", 
                             intern = TRUE)

names_prs_score.afr <- system("ls /medpop/esp2/mesbah/projects/Meta_GWAS/MetaGWAS_N900k/sumHer/prs/scores/afr/*.profile | awk '{print $NF}' | cut -d '.' -f7,8  | sed 's:has::g' | tr '.' '_' ", 
                             intern = TRUE)

list_prs_score.gbr <- system("ls /medpop/esp2/mesbah/projects/Meta_GWAS/MetaGWAS_N900k/sumHer/prs/scores/gbr/*.profile | awk '{print $NF}'", 
                             intern = TRUE)

names_prs_score.gbr <- system("ls /medpop/esp2/mesbah/projects/Meta_GWAS/MetaGWAS_N900k/sumHer/prs/scores/gbr/*.profile | awk '{print $NF}' | cut -d '.' -f7,8  | sed 's:has::g' | tr '.' '_' ", 
                             intern = TRUE)


# PRS with AFR Reference Panel

In [ ]:
d_afr <- fread(list_prs_score.afr[1], header=T, select = c("ID2", "Profile_1"))

str(d_afr)

names(d_afr)

summary(d_afr)

names(d_afr) <- c("ID2", names_prs_score.afr[1])

str(d_afr)

dat_afr <- d_afr; rm(d_afr)

for(i in 2:length(list_prs_score.afr)){
    print(i)
    
    print(list_prs_score.afr[i])
    
    print(names_prs_score.afr[i])
    
    d <- fread(list_prs_score.afr[i], header=T, select = c("ID2", "Profile_1"))
    
    names(d) <- c("ID2", names_prs_score.afr[i])
    
    dat_afr <- merge(dat_afr, d, by="ID2")
    
    rm(d)
    
}

str(dat_afr)

head(dat_afr)


In [ ]:
cor(dat_afr[,c(2:22)])

In [ ]:
heatmap(cor(dat_afr[,c(2:22)]))

# PRS with EUR Reference Panel

In [ ]:
d_eur <- fread(list_prs_score.gbr[1], header=T, select = c("ID2", "Profile_1"))

str(d_eur)

names(d_eur)

summary(d_eur)

names(d_eur) <- c("ID2", names_prs_score.gbr[1])

str(d_eur)

dat_eur <- d_eur; rm(d_eur)

for(i in 2:length(list_prs_score.gbr)){
    print(i)
    
    print(list_prs_score.gbr[i])
    
    print(names_prs_score.gbr[i])
    
    d <- fread(list_prs_score.gbr[i], header=T, select = c("ID2", "Profile_1"))
    
    names(d) <- c("ID2", names_prs_score.gbr[i])
    
    dat_eur <- merge(dat_eur, d, by="ID2")
    
    rm(d)
    
}

str(dat_eur)

head(dat_eur)


In [ ]:
cor(dat_eur[,c(2:22)])

In [ ]:
heatmap(cor(dat_eur[,c(2:22)]))

# CHIP data

In [ ]:
ukb200k_chip <- fread("/medpop/esp2/mesbah/datasets/CHIP/UKBB/ukb450_2023/CH_phenoCovar.ukb200k_N193342.28cols.03_05_2024.tsv.gz", header=T)

ukb250k_chip <- fread("/medpop/esp2/mesbah/datasets/CHIP/UKBB/ukb450_2023/CH_phenoCovar.ukb250k_N243350.28cols.03_05_2024.tsv.gz", header=T)

ukb450k_chip <- as.data.frame(rbind(ukb200k_chip, ukb250k_chip)); rm(ukb200k_chip, ukb250k_chip)

head(ukb450k_chip)  

table(ukb450k_chip$Batch, exclude = NULL)


## AFR REF

In [ ]:
ukb450k_chip_afr <- merge(ukb450k_chip, dat_afr, 
                          by.x="FID", by.y="ID2")

head(ukb450k_chip_afr)
str(ukb450k_chip_afr)
names(ukb450k_chip_afr)

In [ ]:
# heatmap(cor(ukb450k_chip_afr[, c(3:11,29:49)], use="complete", method="spearman"))

(cor(ukb450k_chip_afr[, c(3:6,29:49)], use="complete", method="spearman"))

## EUR REF

In [ ]:
ukb450k_chip_eur <- merge(ukb450k_chip, dat_eur, 
                          by.x="FID", by.y="ID2")

head(ukb450k_chip_eur)
str(ukb450k_chip_eur)
names(ukb450k_chip_eur)

In [ ]:
ls()

In [ ]:
# rm(dat_afr, dat_eur,i,list_prs_score.afr,list_prs_score.gbr,names_prs_score.afr, names_prs_score.gbr)

save(ukb450k_chip, ukb450k_chip_afr, ukb450k_chip_eur, 
     file = "/medpop/esp2/mesbah/projects//Meta_GWAS/MetaGWAS_N900k//sumHer/prs/prs.afr_eur.ukb450k.rda")